In [ ]:
# Importar las librerías necesarias
import pandas as pd
import sqlite3

In [ ]:
# Conexión a la base de datos
dbname = 'TareaIII'
conn = sqlite3.connect(dbname + '.db')

In [ ]:
# Crear cursor y listar tablas en la base de datos
cur = conn.cursor()
cur. execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cur.fetchall())

[('cliente',), ('ventas',), ('producto',), ('detalle',)]


In [ ]:
# Definir una función para ejecutar consultas y mostrar los resultados en formato DataFrame
def read_df(cur,command):
    result = cur.execute(command)
    col_names = list(map(lambda x: x[0], cur.description))
    df_query = pd.DataFrame(result, columns =col_names)
    print(df_query) 

In [ ]:
# Sumar el dinero gastado por cliente en productos
read_df(cur,"""SELECT Nombre, Apellido, SUM(cantidad*precio_unitario) AS Suma_Dinero
               FROM cliente
               LEFT JOIN ventas
               USING (id_cliente)
               LEFT JOIN detalle
               USING (id_boleta)
               LEFT JOIN producto
               USING (id_producto)
               GROUP BY Nombre, Apellido;""")

     Nombre   Apellido  Suma_Dinero
0    Camila       Pina     370410.0
1  Federico    Somaraz     138770.0
2     Italo      Betta          NaN
3  Josefina   Vitulich     306300.0
4      Juan      Perez     540830.0
5     Julio   Magnolfi     284090.0
6   Paulina    Rosales     180050.0
7     Pedro   Gonzalez     145660.0
8     Simon  Sepulveda     321200.0
9   Soledad    Briones     194460.0


In [ ]:
# Clientes que no han realizado ninguna compra 
read_df(cur,"""SELECT cliente.Nombre, cliente.Apellido
               FROM cliente
               LEFT JOIN ventas ON ventas.id_cliente = cliente.id_cliente
               WHERE ventas.id_cliente IS NULL;""")

  Nombre Apellido
0  Italo    Betta


In [ ]:
# Cliente con el mayor monto total de ventas 
read_df(cur,"""SELECT Nombre, Apellido, SUM(cantidad*precio_unitario) AS Suma_Dinero
               FROM cliente
               LEFT JOIN ventas
               USING (id_cliente)
               LEFT JOIN detalle
               USING (id_boleta)
               LEFT JOIN producto
               USING (id_producto)
               GROUP BY Nombre, Apellido
               HAVING SUM(cantidad*precio_unitario)=(
                    SELECT MAX(Suma_Dinero)
                    FROM(
                         SELECT SUM(cantidad*precio_unitario) AS Suma_Dinero
                         FROM cliente
                         LEFT JOIN ventas
                         USING (id_cliente)
                         LEFT JOIN detalle
                         USING (id_boleta)
                         LEFT JOIN producto
                         USING (id_producto)
                         GROUP BY Nombre, Apellido));""")

  Nombre Apellido  Suma_Dinero
0   Juan    Perez       540830
